# Libraries

In [1]:
from pathlib import Path
import os
import sys
import numpy as np
import torch
from torch.utils.data import ConcatDataset, DataLoader
from torchvision import transforms
from PIL import Image
from sklearn.metrics import f1_score

In [ ]:
PROJECT_ROOT = Path.cwd()
PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import agents
from dataloaders.base import RafDB_perm

# Setup

In [ ]:
WEIGHT_PATH = Path(r"")

In [11]:
BATCH_SIZE = 64
NUM_WORKERS = 0
GPUID = [0]

CATEGORY = "gender"
MODEL_TYPE = "resnet"
MODEL_NAME = "TVResNet18_pretrained_freeze"
AGENT_NAME = "ProposedFramework"
SKIP_UNSURE_FOR_GENDER = True
SKIP_UNSURE = SKIP_UNSURE_FOR_GENDER if CATEGORY == "gender" else False

In [ ]:
def patch_resnet_no_download():
    import models.resnet as repo_resnet
    repo_resnet.TVResNet18_pretrained_freeze = lambda: repo_resnet.TVResNet("resnet18", pretrained=False, freeze_initial=True)

def load_state_dict(path):
    state = torch.load(path, map_location="cpu")
    if isinstance(state, dict) and "state_dict" in state:
        state = state["state_dict"]
    if all(k.startswith("module.") for k in state.keys()):
        state = {k[len("module."):]: v for k, v in state.items()}
    return state

patch_resnet_no_download()

agent_config = {
    "lr": 0.001,
    "momentum": 0.0,
    "weight_decay": 0.0,
    "schedule": [1],
    "model_type": MODEL_TYPE,
    "model_name": MODEL_NAME,
    "model_weights": None,
    "out_dim": {"All": 7},
    "optimizer": "Adam",
    "print_freq": 0,
    "gpuid": GPUID,
    "reg_coef": 0.0,
    "prune_amount": 0.0,
    "target_acc": 0.85,
    "alpha": 0.05,
    "init_prune_rate": 0.95,
    "inc_prune_rate": 0.50,
    "prune_retrain_epochs": 0,
    "lambda_bias": 1.0,
}

agent_factory = getattr(agents.customization, AGENT_NAME)
agent = agent_factory(agent_config)
state_dict = load_state_dict(WEIGHT_PATH)
agent.model.load_state_dict(state_dict, strict=True)
agent.eval()

Loaded: resnet_ProposedFramework_42.08080823957882_gender_augmented_epoch18_end_2026-07-01_18-14-08.pth


C:\Users\gassa\AppData\Local\Temp\ipykernel_25328\3051780619.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(path, map_location="cpu")


# Load Data

In [13]:
_, val_datasets, _ = RafDB_perm(CATEGORY, train_aug=False, skip_unsure=SKIP_UNSURE)
task_names = sorted(val_datasets.keys(), key=int)
val_all = ConcatDataset([val_datasets[name] for name in task_names])
val_loader = DataLoader(val_all, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print("tasks:", task_names)
print("validation samples:", len(val_all))

Loading cached attributes...


Reading classes: 100%|██████████| 15339/15339 [00:00<00:00, 667099.70line/s]
c:\Users\gassa\Kaiwen\UM\2025_2026\Semester_1\Project\.GitCode\Benchmarkwith_RAF-DB\dataloaders\wrapper.py:23: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on G

tasks: ['1', '2']
validation samples: 2869


# Eval

In [19]:
def evaluate(agent, loader):
    device = next(agent.model.parameters()).device
    y_true, y_pred = [], []
    group_total = {}
    group_correct = {}

    agent.eval()
    with torch.no_grad():
        for inputs, targets, tasks in loader:
            inputs = inputs.to(device)
            targets = targets.to(device)
            logits = agent.predict(inputs)["All"]
            preds = logits.argmax(dim=1)

            y_true.extend(targets.cpu().tolist())
            y_pred.extend(preds.cpu().tolist())

            correct = preds.eq(targets).cpu().tolist()
            for task, is_correct in zip(tasks, correct):
                task = str(task)
                group_total[task] = group_total.get(task, 0) + 1
                group_correct[task] = group_correct.get(task, 0) + int(is_correct)

    overall_acc = np.mean(np.array(y_true) == np.array(y_pred))
    weighted_f1 = f1_score(y_true, y_pred, average="weighted")

    group_acc = {task: group_correct[task] / group_total[task] for task in group_total}
    group_values = list(group_acc.values())
    acc_gap = max(group_values) - min(group_values)
    fairness_ratio = min(group_values) / max(group_values) if max(group_values) > 0 else 0.0

    chi2 = 0.0
    for task in group_total:
        expected_correct = group_total[task] * overall_acc
        if expected_correct > 0:
            observed_correct = group_correct[task]
            chi2 += ((observed_correct - expected_correct) ** 2) / expected_correct

    return overall_acc, weighted_f1, group_total, group_correct, group_acc, acc_gap, fairness_ratio, chi2

In [20]:
overall_acc, weighted_f1, group_total, group_correct, group_acc, acc_gap, fairness_ratio, chi2 = evaluate(agent, val_loader)

nonzero_params = agent.count_parameter()
total_params = sum(p.numel() for p in agent.model.parameters())
sparsity = 1.0 - (nonzero_params / total_params)

In [22]:
print(f"Overall accuracy:   {overall_acc * 100:.2f}%")
print(f"Fairness Score:     {fairness_ratio:.4f}")
print(f"Params:             {nonzero_params:,} / {total_params:,}")
print(f"Pruned:             {sparsity * 100:.2f}%")

print("\nGroup accuracy:")
for task in sorted(group_total.keys(), key=int):
    print(
        f"  Task {task}: {group_acc[task] * 100:6.2f}% "
        f"({group_correct[task]}/{group_total[task]})"
    )

Overall accuracy:   74.90%
Fairness Score:     0.9581
Params:             1,618,964 / 11,180,103
Pruned:             85.52%

Group accuracy:
  Task 1:  73.10% (913/1249)
  Task 2:  76.30% (1236/1620)


# Inference

In [24]:
EMOTION_LABELS = [
    "Surprise",
    "Fear",
    "Disgust",
    "Happiness",
    "Sadness",
    "Anger",
    "Neutral",
]

IMAGE_PATHS = [
    # "RafDB/basic/Image/aligned/test_0001_aligned.jpg",
    'notebooks/images/Male_Neutral.jpeg',
]

infer_transform = transforms.Compose([
    transforms.Resize((100, 100)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def resolve_path(path):
    path = Path(path)
    return path if path.is_absolute() else PROJECT_ROOT / path

def get_inference_paths():
    paths = [resolve_path(path) for path in IMAGE_PATHS]
    missing = [path for path in paths if not path.exists()]

    if missing:
        raise FileNotFoundError("Missing image file(s): " + ", ".join(str(path) for path in missing))

    return paths

def predict_image(path):
    image = Image.open(path).convert("RGB")
    tensor = infer_transform(image).unsqueeze(0)
    device = next(agent.model.parameters()).device

    agent.eval()
    with torch.no_grad():
        logits = agent.predict(tensor.to(device))["All"]
        probabilities = torch.softmax(logits, dim=1)[0].cpu()

    pred_label = int(probabilities.argmax())
    confidence = float(probabilities[pred_label])
    return pred_label, confidence, probabilities


In [25]:
for image_path in get_inference_paths():
    pred_label, confidence, probabilities = predict_image(image_path)
    print("Image:", image_path)
    print("Predicted label:", EMOTION_LABELS[pred_label])
    print(f"Confidence: {confidence:.4f}")
    print("Class probabilities:")
    for label, prob in zip(EMOTION_LABELS, probabilities.tolist()):
        print(f"  {label:>10}: {prob:.4f}")
    print()

Image: c:\Users\gassa\Kaiwen\UM\2025_2026\Semester_1\Project\.GitCode\Benchmarkwith_RAF-DB\notebooks\images\Male_Neutral.jpeg
Predicted label: Neutral
Confidence: 0.3211
Class probabilities:
    Surprise: 0.0825
        Fear: 0.0559
     Disgust: 0.0970
   Happiness: 0.1421
     Sadness: 0.2190
       Anger: 0.0824
     Neutral: 0.3211

